# Scientific Reports successor — Step 6

Dimensionless waveform susceptibility and critical-anisotropy inversion. The notebook consumes passing Step 4 and Step 5 results, executes the six native arteries and stops before crossed waveform experiments.

In [ ]:
from pathlib import Path
import os, subprocess, sys
IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'successor/scirep-waveform-susceptibility'
for key in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'):
    os.environ[key] = '1'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    repo_root = Path('/content/picoNewton')
    if not repo_root.exists():
        subprocess.run(['git','clone','https://github.com/khalid-saqr/picoNewton.git',str(repo_root)],check=True)
    subprocess.run(['git','-C',str(repo_root),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(repo_root),'checkout','-B',BRANCH,f'origin/{BRANCH}'],check=True)
    study_root = Path('/content/drive/MyDrive/picoNewton_susceptibility')
else:
    repo_root = Path.cwd()
    while repo_root != repo_root.parent and not (repo_root/'picoNewton_v3').exists():
        repo_root = repo_root.parent
    study_root = repo_root/'piconewton_susceptibility_outputs'
print({'repo_root':str(repo_root),'study_root':str(study_root),'colab':IN_COLAB})

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','-e',str(repo_root/'picoNewton_v3')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','-e',str(repo_root/'piconewton_susceptibility')],check=True)

## Resolve prior gates

A clean Drive reconstructs Steps 2–5. Existing outputs are reused only after the fail-closed validators inspect their manifests and checksums.

In [ ]:
step2_root = study_root/'bootstrap'/'step2'
step3_root = study_root/'step3_parent_continuity'
step4_root = study_root/'step4_perturbation'
step5_root = study_root/'step5_harmonic_kernel'
step6_root = study_root/'step6_susceptibility'
if not (step2_root/'completion_gate.json').exists():
    subprocess.run(['piconewton-susceptibility-bootstrap','--repo-root',str(repo_root),'--storage','local','--local-root',str(study_root)],check=True)
if not (step3_root/'step3_manifest.json').exists():
    subprocess.run(['piconewton-susceptibility-step3','--step2-root',str(step2_root),'--output',str(step3_root),'--profile','publication'],check=True)
if not (step4_root/'step4_manifest.json').exists():
    subprocess.run(['piconewton-susceptibility-step4','--step3-root',str(step3_root),'--output',str(step4_root),'--profile','publication'],check=True)
if not (step5_root/'step5_manifest.json').exists():
    subprocess.run(['piconewton-susceptibility-step5','--step4-root',str(step4_root),'--output',str(step5_root),'--profile','publication'],check=True)
print({'step4_root':str(step4_root),'step5_root':str(step5_root),'step6_root':str(step6_root)})

## Execute Step 6 publication profile

In [ ]:
subprocess.run([
    'piconewton-susceptibility-step6',
    '--step5-root',str(step5_root),
    '--step4-root',str(step4_root),
    '--output',str(step6_root),
    '--profile','publication'
],check=True)

In [ ]:
import json, pandas as pd
manifest=json.loads((step6_root/'step6_manifest.json').read_text())
native=pd.read_csv(step6_root/'native_susceptibility.csv')
critical=pd.read_csv(step6_root/'critical_anisotropy.csv')
print(json.dumps(manifest['gates'],indent=2,sort_keys=True))
display(native[['artery_name','phi_rms','phi_peak_abs','force2_rms_n_per_epsilon2','predicted_rms_at_epsilon_0p1_n']])
display(critical[['artery_name','metric','benchmark_pn','perturbative_epsilon_critical','validated_domain_max','exact_metric_at_domain_max_pn','status']])

## Stop boundary

A passing Step 6 manifest authorises Step 7. This notebook does not cross vessel and waveform cases, perform harmonic ablations, fit a reduced law, or vary the constitutive tensor away from the reciprocal path.